In [300]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType, IntegerType
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("PySpark Ex1").getOrCreate()

In [301]:
# Target schema = what we want AFTER cleaning (documentation + final enforcement).
target_schema = StructType([
    StructField("customer_id",    IntegerType(), False),
    StructField("full_name",      StringType(),  True),
    StructField("email",          StringType(),  True),
    StructField("signup_date",    DateType(),    True),
    StructField("age",            IntegerType(), True),
    StructField("country",        StringType(),  True),
    StructField("lifetime_value", DoubleType(),  True),
])

raw_cols = ["customer_id", "full_name", "email", "signup_date", "age", "country", "lifetime_value"]

# Read as strings so nothing is lost.
string_schema = StructType([StructField(c, StringType(), True) for c in raw_cols])
df = spark.read.option("header", True).schema(string_schema).csv("dirty_customers.csv")

df.show(20, truncate=False)

+-----------+----------------+--------------------------+-----------+----+--------------+--------------+
|customer_id|full_name       |email                     |signup_date|age |country       |lifetime_value|
+-----------+----------------+--------------------------+-----------+----+--------------+--------------+
|1001       |  Alice Johnson |alice.johnson@email.com   |2023-01-15 |34  |USA           |$1,250.50     |
|1002       |BOB SMITH       |BOB.SMITH@EMAIL.COM       |01/22/2023 |29  |usa           |980.00        |
|1003       |charlie brown   |charlie.brown(at)email.com|2023-02-03 |41  |United States |2,100.75      |
|1004       |Dana White      |dana.white@email.com      |2023/03/10 |-5  |Canada        |$540.00       |
|1005       |                |NULL                      |2023-03-11 |NULL|canada        |NULL          |
|1006       |Eve Adams       |eve.adams@email.com       |2023-04-01 |27  |CA            |1500          |
|1002       |BOB SMITH       |BOB.SMITH@EMAIL.COM      

In [302]:
print("Row count:", df.count())
print("Schema:")
df.printSchema()

# Null / empty counts per column
df.select([
    F.count(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), c)).alias(c)
    for c in df.columns
]).show()

# Distinct values for the categorical column
df.select("country").distinct().show(truncate=False)
df.select("signup_date").distinct().show(truncate=False)

# Duplicate detection
print("Total rows vs distinct customer_id:",
      df.count(), df.select("customer_id").distinct().count())

Row count: 15
Schema:
root
 |-- customer_id: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- age: string (nullable = true)
 |-- country: string (nullable = true)
 |-- lifetime_value: string (nullable = true)

+-----------+---------+-----+-----------+---+-------+--------------+
|customer_id|full_name|email|signup_date|age|country|lifetime_value|
+-----------+---------+-----+-----------+---+-------+--------------+
|          0|        1|    2|          1|  1|      1|             1|
+-----------+---------+-----+-----------+---+-------+--------------+

+--------------+
|country       |
+--------------+
|uk            |
|United States |
|CA            |
|canada        |
|U.S.A.        |
|USA           |
|UK            |
|Canada        |
|usa           |
|United Kingdom|
|NULL          |
+--------------+

+-----------+
|signup_date|
+-----------+
|2023-03-11 |
|2023-06-02 |
|2023-05-20 |
|0

In [303]:
# String treatment

text_cols = ["full_name", "email", "country"]

for c in text_cols:
    df = df.withColumn(c, F.trim(F.col(c)))
    df = df.withColumn(c, F.when(F.col(c) == "", None)
                            .otherwise(F.col(c)))

# Lower-case the values
df = df.withColumn("full_name", F.initcap(F.col("full_name")))
df = df.withColumn("email", F.lower(F.col("email")))
df = df.withColumn("country", F.lower(F.col("country")))

df.show(20, truncate=False)

+-----------+-------------+--------------------------+-----------+----+--------------+--------------+
|customer_id|full_name    |email                     |signup_date|age |country       |lifetime_value|
+-----------+-------------+--------------------------+-----------+----+--------------+--------------+
|1001       |Alice Johnson|alice.johnson@email.com   |2023-01-15 |34  |usa           |$1,250.50     |
|1002       |Bob Smith    |bob.smith@email.com       |01/22/2023 |29  |usa           |980.00        |
|1003       |Charlie Brown|charlie.brown(at)email.com|2023-02-03 |41  |united states |2,100.75      |
|1004       |Dana White   |dana.white@email.com      |2023/03/10 |-5  |canada        |$540.00       |
|1005       |NULL         |NULL                      |2023-03-11 |NULL|canada        |NULL          |
|1006       |Eve Adams    |eve.adams@email.com       |2023-04-01 |27  |ca            |1500          |
|1002       |Bob Smith    |bob.smith@email.com       |01/22/2023 |29  |usa        

In [304]:
# Emails treatment

df = (df
    .withColumn("email", F.regexp_replace("email", r"\s*\(at\)\s*", "@"))
    .withColumn("email", F.regexp_replace("email", r"\s*[\(\[]dot[\)\]]\s*", "."))
    .withColumn("email", F.regexp_replace("email", r"@{2,}", "@"))
    .withColumn("email", F.regexp_replace("email", r"\s+", ""))
)

df.select("customer_id", "email").show(20, truncate=False)

+-----------+-----------------------+
|customer_id|email                  |
+-----------+-----------------------+
|1001       |alice.johnson@email.com|
|1002       |bob.smith@email.com    |
|1003       |charlie.brown@email.com|
|1004       |dana.white@email.com   |
|1005       |NULL                   |
|1006       |eve.adams@email.com    |
|1002       |bob.smith@email.com    |
|1007       |frank.miller@email.com |
|1008       |grace.lee@email.com    |
|1009       |NULL                   |
|1010       |ivy.chen@email.com     |
|1011       |jack.black@email.com   |
|1003       |charlie.brown@email.com|
|1012       |karen.page@email       |
|1013       |leo.messi@email.com    |
+-----------+-----------------------+



In [305]:
# Data treatment
df = df.withColumn("signup_date", F.coalesce(
    F.expr("try_to_timestamp(replace(signup_date,'/','-'), 'yyyy-MM-dd')"),
    F.expr("try_to_timestamp(replace(signup_date,'/','-'), 'MM-dd-yyyy')"),
    F.expr("try_to_timestamp(replace(signup_date,'/','-'), 'dd-MM-yyyy')"),
).cast("date"))

df.select("customer_id", "signup_date").show(truncate=False)

+-----------+-----------+
|customer_id|signup_date|
+-----------+-----------+
|1001       |2023-01-15 |
|1002       |2023-01-22 |
|1003       |2023-02-03 |
|1004       |2023-03-10 |
|1005       |2023-03-11 |
|1006       |2023-04-01 |
|1002       |2023-01-22 |
|1007       |2023-04-15 |
|1008       |2023-05-15 |
|1009       |2023-05-20 |
|1010       |2023-06-02 |
|1011       |2023-06-18 |
|1003       |2023-02-03 |
|1012       |2023-07-09 |
|1013       |NULL       |
+-----------+-----------+



In [306]:
# Stardard for age column

age_int = F.expr("try_cast(age as int)")

df = df.withColumn("age",
                   F.when(age_int < 0, 0)
                   .when(age_int.isNull(), 0)
                   .when(age_int > 120, 0)
                   .otherwise(age_int))

df.show(20, truncate=False)

+-----------+-------------+-----------------------+-----------+---+--------------+--------------+
|customer_id|full_name    |email                  |signup_date|age|country       |lifetime_value|
+-----------+-------------+-----------------------+-----------+---+--------------+--------------+
|1001       |Alice Johnson|alice.johnson@email.com|2023-01-15 |34 |usa           |$1,250.50     |
|1002       |Bob Smith    |bob.smith@email.com    |2023-01-22 |29 |usa           |980.00        |
|1003       |Charlie Brown|charlie.brown@email.com|2023-02-03 |41 |united states |2,100.75      |
|1004       |Dana White   |dana.white@email.com   |2023-03-10 |0  |canada        |$540.00       |
|1005       |NULL         |NULL                   |2023-03-11 |0  |canada        |NULL          |
|1006       |Eve Adams    |eve.adams@email.com    |2023-04-01 |27 |ca            |1500          |
|1002       |Bob Smith    |bob.smith@email.com    |2023-01-22 |29 |usa           |980.00        |
|1007       |Frank M

In [307]:
# Standard for country
df.select("country").distinct().show(truncate=False)   # inspect first

c = F.lower(F.trim(F.col("country")))

df = df.withColumn(
    "country",
    F.when(c.isin("usa", "united states", "u.s.a.", "us"), "United States")
     .when(c.isin("uk", "united kingdom", "u.k."), "United Kingdom")
     .when(c.isin("canada", "ca"), "Canada")
     .when(c.isNull(), "Unknown")
     .otherwise(F.initcap(c))
)

df.select("country").distinct().show(truncate=False)


+--------------+
|country       |
+--------------+
|uk            |
|u.s.a.        |
|canada        |
|ca            |
|united states |
|united kingdom|
|usa           |
|NULL          |
+--------------+

+--------------+
|country       |
+--------------+
|United States |
|Unknown       |
|Canada        |
|United Kingdom|
+--------------+



In [308]:
# Standard for lifetime_value

df = df.withColumn(
    "lifetime_value",
    F.regexp_replace(F.col("lifetime_value"), r"[\$,]|GBP|EUR|USD",  "")
)

df = df.withColumn(
    "lifetime_value",
    F.expr("try_cast(lifetime_value as double)")
)


df = df.withColumn(
    "lifetime_value",
    F.when(F.col("lifetime_value").isNull(), 0)
    .otherwise(F.col("lifetime_value"))
)

df.select("lifetime_value").show(truncate=False)

+--------------+
|lifetime_value|
+--------------+
|1250.5        |
|980.0         |
|2100.75       |
|540.0         |
|0.0           |
|1500.0        |
|980.0         |
|0.0           |
|720.0         |
|640.25        |
|2890.0        |
|1030.0        |
|2100.75       |
|415.0         |
|1875.4        |
+--------------+



In [312]:
df.orderBy("customer_id").show(truncate=False)

+-----------+-------------+-----------------------+-----------+---+--------------+--------------+
|customer_id|full_name    |email                  |signup_date|age|country       |lifetime_value|
+-----------+-------------+-----------------------+-----------+---+--------------+--------------+
|1001       |Alice Johnson|alice.johnson@email.com|2023-01-15 |34 |United States |1250.5        |
|1002       |Bob Smith    |bob.smith@email.com    |2023-01-22 |29 |United States |980.0         |
|1002       |Bob Smith    |bob.smith@email.com    |2023-01-22 |29 |United States |980.0         |
|1003       |Charlie Brown|charlie.brown@email.com|2023-02-03 |41 |United States |2100.75       |
|1003       |Charlie Brown|charlie.brown@email.com|2023-02-03 |41 |United States |2100.75       |
|1004       |Dana White   |dana.white@email.com   |2023-03-10 |0  |Canada        |540.0         |
|1005       |NULL         |NULL                   |2023-03-11 |0  |Canada        |0.0           |
|1006       |Eve Ada

In [315]:
df = df.dropDuplicates()

In [316]:
df.orderBy("customer_id").show(truncate=False)

+-----------+-------------+-----------------------+-----------+---+--------------+--------------+
|customer_id|full_name    |email                  |signup_date|age|country       |lifetime_value|
+-----------+-------------+-----------------------+-----------+---+--------------+--------------+
|1001       |Alice Johnson|alice.johnson@email.com|2023-01-15 |34 |United States |1250.5        |
|1002       |Bob Smith    |bob.smith@email.com    |2023-01-22 |29 |United States |980.0         |
|1003       |Charlie Brown|charlie.brown@email.com|2023-02-03 |41 |United States |2100.75       |
|1004       |Dana White   |dana.white@email.com   |2023-03-10 |0  |Canada        |540.0         |
|1005       |NULL         |NULL                   |2023-03-11 |0  |Canada        |0.0           |
|1006       |Eve Adams    |eve.adams@email.com    |2023-04-01 |27 |Canada        |1500.0        |
|1007       |Frank Miller |frank.miller@email.com |2023-04-15 |0  |United States |0.0           |
|1008       |Grace L